# BAPS cohort plots

Converted from `plot_baps.py`. Run cells in order. The last cell starts the interactive scorer / plots.

Data live under `experiments/LevelN/Results_CSV` and `Results_xdf` (not in git). Outputs go to `analysis/outputs/`.


In [ ]:
# Run from repo root or from analysis/. This replaces __file__ used by the .py scripts.
from pathlib import Path
import os, sys
cwd = Path.cwd().resolve()
if (cwd / "analysis").is_dir() and (cwd / "experiments").is_dir():
    REPO_ROOT = cwd
elif cwd.name == "analysis":
    REPO_ROOT = cwd.parent
elif cwd.name == "notebooks":
    REPO_ROOT = cwd.parents[1]
else:
    REPO_ROOT = cwd
_ANALYSIS_DIR = REPO_ROOT / "analysis"
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)
%matplotlib inline


Standalone BAPS Plot Generator
================================
Run this script from ANY Python environment that has matplotlib working:

    python3 plot_baps_results.py BAPS_Subject2_results.csv

It reads all CSVs saved by behavioral_attention_analysis.py and generates
all 8 plots without depending on the analysis environment.

Required file (mandatory):
    BAPS_Subject{N}_results.csv

Optional files (auto-detected from same folder):
    BAPS_Subject{N}_trial_rt.csv  (may include At_distraction=1 on distraction trials)
    BAPS_Subject{N}_debt.csv
    BAPS_Subject{N}_cost.csv
    BAPS_Subject{N}_shapes.csv

Cohort RT at distraction (all subjects with updated trial_rt):

    python3 plot_baps_results.py --cohort-plot4 --data-dir /path/to/GoNoGoTests

Writes Analysis_Results/cohort_trajectory/plot4_cohort_rt_at_distraction.png


In [ ]:
import sys
import os
import re
import glob
import argparse

# Force Agg backend before anything else touches matplotlib.
# This works in all environments (base Python, venvs, conda).
os.environ['MPLBACKEND'] = 'Agg'
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd

# ── Color palette (same as analysis script) ──────────────────────────────────
COND_COLORS = {
    'Baseline':                '#4C9BE8',
    'Distraction+Response':    '#E84C4C',
    'Distraction+NoResponse':  '#E8A94C',
}
# Display-only labels for legends/titles — condition keys above stay as-is
# since they're also used to look up data, but "NoResponse" reads as
# "Glance" to match the paper's condition naming (glance-only, no reply).
COND_DISPLAY = {
    'Baseline':               'Baseline',
    'Distraction+Response':   'Distraction+Response',
    'Distraction+NoResponse': 'Distraction+Glance',
}
SET_COLORS = {1: '#4C9BE8', 2: '#E84C4C', 3: '#E8A94C'}
SHAPE_COLORS = {
    'V': '#2ECC71', 'U': '#F39C12', 'U_slow': '#E67E22',
    'L': '#E74C3C', 'L_improving': '#C0392B', 'no_recovery': '#7F8C8D',
}
SHAPE_LABELS = {
    'V': 'V-shape\n(rapid rebound)', 'U': 'U-shape\n(gradual)',
    'U_slow': 'U-slow\n(effortful)', 'L': 'L-shape\n(no recovery)',
    'L_improving': 'L-improve\n(trending)', 'no_recovery': 'No recovery',
}

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'font.size': 12})


def save(fig, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"   Saved: {path}")


def _backfill_at_distraction_from_behavioral(trial_df, subject_id, data_dir):
    """
    Add At_distraction to a trial_rt frame using Level*/Results_CSV behavioral files
    (same row order as BAPS export). Returns trial_df unchanged on failure.
    """
    if trial_df is None or trial_df.empty:
        return trial_df
    if 'At_distraction' in trial_df.columns:
        return trial_df
    _here = str(_ANALYSIS_DIR)
    if _here not in sys.path:
        sys.path.insert(0, _here)
    try:
        from regenerate_trajectory_plots import load_behavioral_data
    except ImportError:
        return trial_df

    beh = load_behavioral_data(subject_id, data_dir)
    if not beh:
        return trial_df

    flags = []
    for key in sorted(beh.keys()):
        _lvl, st = key
        for df in beh[key]:
            stim_c = next((c for c in df.columns if str(c).lower() == 'stim_time'), None)
            dfr = df.reset_index(drop=True)
            for i in range(len(dfr)):
                row = dfr.iloc[i]
                at = 0
                if stim_c is not None and st in (2, 3):
                    v = row[stim_c]
                    if pd.notna(v) and str(v).strip().lower() not in ('', 'nan', 'none'):
                        at = 1
                flags.append(at)

    if len(flags) != len(trial_df):
        print(
            f"   Warning: Subject {subject_id} trial_rt rows ({len(trial_df)}) "
            f"!= behavioral rows ({len(flags)}); cannot add At_distraction"
        )
        return trial_df

    out = trial_df.copy()
    out['At_distraction'] = np.asarray(flags, dtype=int)
    return out


# ── Plot 1: BAPS_s by condition ───────────────────────────────────────────────
def plot1_baps_by_condition(df, out_dir):
    conditions = list(COND_COLORS)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
    fig.suptitle('BAPS_s by Task and Distraction Type', fontsize=13, fontweight='bold')
    for ax_idx, level_id in enumerate([1, 2, 3]):
        ax = axes[ax_idx]
        level_data = df[df['Level'] == level_id]
        tasks = level_data['Task'].unique()
        x = np.arange(len(tasks))
        for c_idx, cond in enumerate(conditions):
            cond_data = level_data[level_data['Condition'] == cond]
            vals = [cond_data[cond_data['Task'] == t]['BAPS_s'].values[0]
                    if len(cond_data[cond_data['Task'] == t]) else np.nan
                    for t in tasks]
            if len(x):
                ax.bar(x + c_idx * 0.25, vals, 0.25,
                       label=cond, color=COND_COLORS[cond], alpha=0.85,
                       edgecolor='white', linewidth=0.8)
        ax.axhline(100, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
        ax.set_title(f'Level {level_id}', fontweight='bold')
        ax.set_xlabel('Task')
        if ax_idx == 0:
            ax.set_ylabel('BAPS_s (%)')
        if len(tasks):
            ax.set_xticks(x + 0.25)
            ax.set_xticklabels(tasks, rotation=20, ha='right', fontsize=8)
        ax.set_ylim(0, 160)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.legend(fontsize=7)
    plt.tight_layout()
    save(fig, os.path.join(out_dir, 'plot1_baps_by_condition.png'))


# ── Plot 1b: BAPS_3CC by condition ───────────────────────────────────────────
def plot1b_baps3cc_by_condition(df, out_dir):
    if 'BAPS_3CC' not in df.columns:
        return
    conditions = list(COND_COLORS)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
    fig.suptitle('BAPS_3CC by Task and Distraction Type', fontsize=13, fontweight='bold')
    for ax_idx, level_id in enumerate([1, 2, 3]):
        ax = axes[ax_idx]
        level_data = df[df['Level'] == level_id]
        tasks = level_data['Task'].unique()
        x = np.arange(len(tasks))
        for c_idx, cond in enumerate(conditions):
            cond_data = level_data[level_data['Condition'] == cond]
            vals = [cond_data[cond_data['Task'] == t]['BAPS_3CC'].values[0]
                    if len(cond_data[cond_data['Task'] == t]) else np.nan
                    for t in tasks]
            if len(x):
                ax.bar(x + c_idx * 0.25, vals, 0.25,
                       label=cond, color=COND_COLORS[cond], alpha=0.85,
                       edgecolor='white', linewidth=0.8)
        ax.axhline(100, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
        ax.set_title(f'Level {level_id}', fontweight='bold')
        ax.set_xlabel('Task')
        if ax_idx == 0:
            ax.set_ylabel('BAPS_3CC (%)')
        if len(tasks):
            ax.set_xticks(x + 0.25)
            ax.set_xticklabels(tasks, rotation=20, ha='right', fontsize=8)
        ax.set_ylim(0, 160)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.legend(fontsize=7)
    plt.tight_layout()
    save(fig, os.path.join(out_dir, 'plot1b_baps3cc_by_condition.png'))


# ── Plot 2: Heatmap ───────────────────────────────────────────────────────────
def plot2_heatmap(df, out_dir):
    pivot = df.pivot_table(values='BAPS_s', index='Participant',
                           columns=['Level', 'Condition', 'Task'], aggfunc='mean')
    if pivot.empty:
        return
    norm = TwoSlopeNorm(vmin=30, vcenter=100, vmax=160)
    fig, ax = plt.subplots(figsize=(max(14, pivot.shape[1] * 0.9),
                                    max(5, pivot.shape[0] * 0.7)))
    im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn', norm=norm)
    plt.colorbar(im, ax=ax, label='BAPS_s (%)', shrink=0.8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'P{p}' for p in pivot.index], fontsize=9)
    col_labels = [f'L{l}\n{c[:3]}\n{t[:5]}' for l, c, t in pivot.columns]
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(col_labels, fontsize=7, rotation=45, ha='right')
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                        fontsize=6, color='black' if 60 < val < 130 else 'white')
    ax.set_title('BAPS_s Heatmap (green > 100 = above baseline)', fontweight='bold')
    plt.tight_layout()
    save(fig, os.path.join(out_dir, 'plot2_individual_heatmap.png'))


# ── Plot 3: Recovery time bar chart ──────────────────────────────────────────
def plot3_recovery_time(df, out_dir):
    dist_data = df[df['Condition'] != 'Baseline'].copy()
    if dist_data.empty or 'Recovery_s' not in dist_data.columns:
        return
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    fig.suptitle('Recovery Time by Task and Distraction Type', fontsize=15, fontweight='bold')
    for ax_idx, level_id in enumerate([1, 2, 3]):
        ax = axes[ax_idx]
        level_data = dist_data[dist_data['Level'] == level_id]
        tasks = level_data['Task'].unique()
        x = np.arange(len(tasks))
        for c_idx, cond in enumerate(['Distraction+Response', 'Distraction+NoResponse']):
            cond_data = level_data[level_data['Condition'] == cond]
            vals = [cond_data[cond_data['Task'] == t]['Recovery_s'].values[0]
                    if len(cond_data[cond_data['Task'] == t]) else np.nan
                    for t in tasks]
            if len(x):
                bars = ax.bar(x + c_idx * 0.35, vals, 0.35,
                              label=cond, color=COND_COLORS[cond], alpha=0.85,
                              edgecolor='white')
                for bar, val in zip(bars, vals):
                    if val and not (val != val):  # skip nan
                        ax.text(bar.get_x() + bar.get_width() / 2,
                                bar.get_height() + 0.1,
                                f'{val:.1f}s', ha='center', va='bottom',
                                fontsize=9, fontweight='bold')
        ax.set_title(f'Level {level_id}', fontweight='bold', fontsize=13)
        ax.set_xlabel('Task', fontsize=11, fontweight='bold')
        if ax_idx == 0:
            ax.set_ylabel('Recovery Time (s)', fontsize=11, fontweight='bold')
        ax.tick_params(axis='y', labelsize=10)
        if len(tasks):
            ax.set_xticks(x + 0.175)
            ax.set_xticklabels(tasks, rotation=25, ha='right', fontsize=10, fontweight='bold')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.legend(fontsize=9, framealpha=0.8)
    plt.tight_layout()
    save(fig, os.path.join(out_dir, 'plot3_recovery_time.png'))


# ── Plot 3b: Trials to 3CC bar chart ─────────────────────────────────────────
def plot3b_trials_3cc(df, out_dir):
    if 'Trials_to_3CC' not in df.columns:
        return
    dist_data = df[df['Condition'] != 'Baseline'].copy()
    if dist_data.empty:
        return
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    fig.suptitle('Trials to 3 Consecutive Correct (Recovery Speed)', fontsize=15, fontweight='bold')
    for ax_idx, level_id in enumerate([1, 2, 3]):
        ax = axes[ax_idx]
        level_data = dist_data[dist_data['Level'] == level_id]
        tasks = level_data['Task'].unique()
        x = np.arange(len(tasks))
        for c_idx, cond in enumerate(['Distraction+Response', 'Distraction+NoResponse']):
            cond_data = level_data[level_data['Condition'] == cond]
            vals = [cond_data[cond_data['Task'] == t]['Trials_to_3CC'].values[0]
                    if len(cond_data[cond_data['Task'] == t]) else np.nan
                    for t in tasks]
            if len(x):
                bars = ax.bar(x + c_idx * 0.35, vals, 0.35,
                              label=cond, color=COND_COLORS[cond], alpha=0.85,
                              edgecolor='white')
                for bar, val in zip(bars, vals):
                    if val and not (val != val):  # skip nan
                        ax.text(bar.get_x() + bar.get_width() / 2,
                                bar.get_height() + 0.1,
                                f'{val:.1f}', ha='center', va='bottom',
                                fontsize=9, fontweight='bold')
        ax.set_title(f'Level {level_id}', fontweight='bold', fontsize=13)
        ax.set_xlabel('Task', fontsize=11, fontweight='bold')
        if ax_idx == 0:
            ax.set_ylabel('Mean Trials to 3CC', fontsize=11, fontweight='bold')
        ax.tick_params(axis='y', labelsize=10)
        if len(tasks):
            ax.set_xticks(x + 0.175)
            ax.set_xticklabels(tasks, rotation=25, ha='right', fontsize=10, fontweight='bold')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.legend(fontsize=9, framealpha=0.8)
    plt.tight_layout()
    save(fig, os.path.join(out_dir, 'plot3b_trials_to_3cc.png'))


# ── Plot 4: Performance trajectory ───────────────────────────────────────────
def plot4_trajectory(trial_df, out_dir):
    """Scatter RT at distraction trials only (no lines). Uses At_distraction from trial_rt."""
    if trial_df is None or trial_df.empty:
        print("   Skipping plot4 (no trial RT data)")
        return
    has_mark = 'At_distraction' in trial_df.columns
    if not has_mark:
        print(
            "   Note: trial_rt has no At_distraction column — "
            "showing all trials as points (re-run behavioral export for distraction-only)."
        )

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    cond_map = {1: 'Baseline', 2: 'Distraction+Response', 3: 'Distraction+NoResponse'}
    for ax_idx, level_id in enumerate([1, 2, 3]):
        ax = axes[ax_idx]
        for set_id in [1, 2, 3]:
            grp = trial_df[(trial_df['Level'] == level_id) & (trial_df['Set'] == set_id)].copy()
            if grp.empty:
                continue
            grp = grp.reset_index(drop=True)
            seq = np.arange(len(grp), dtype=float)
            if has_mark:
                m = grp['At_distraction'].values == 1
            else:
                m = np.ones(len(grp), dtype=bool)
            if not np.any(m):
                continue
            cond = cond_map.get(set_id, str(set_id))
            color = COND_COLORS.get(cond, '#95A5A6')
            ax.scatter(
                seq[m],
                grp['RT'].values[m],
                c=color,
                s=32,
                alpha=0.78,
                edgecolors='white',
                linewidths=0.35,
                label=COND_DISPLAY.get(cond, cond),
            )
        ax.set_title(f'Level {level_id}', fontweight='bold', fontsize=15)
        ax.set_xlabel('Trial (concatenated session)', fontsize=13, fontweight='bold')
        if ax_idx == 0:
            ylab = 'RT at distraction trial (s)' if has_mark else 'RT (s)'
            ax.set_ylabel(ylab, fontsize=13, fontweight='bold')
        ax.tick_params(axis='both', labelsize=11)
        ax.grid(alpha=0.3, linestyle='--')
        ax.legend(fontsize=11, markerscale=1.6)
    plt.tight_layout()
    save(fig, os.path.join(out_dir, 'plot4_performance_trajectory.png'))


def plot4_cohort_rt_at_distraction(data_dir, out_dir):
    """
    Cohort mean RT (seconds) on distraction-marked trials, by level and set (2 vs 3).
    Each participant contributes one mean RT per (Level, Set) across tasks; cohort
    mean ± SEM is shown as compact grouped bars.
    """
    paths = sorted(glob.glob(os.path.join(data_dir, 'BAPS_Subject*_trial_rt.csv')))
    frames = []
    for p in paths:
        m = re.search(r'Subject(\d+)_trial_rt\.csv$', os.path.basename(p))
        if not m:
            continue
        pid = int(m.group(1))
        tdf = pd.read_csv(p)
        if 'At_distraction' not in tdf.columns:
            tdf = _backfill_at_distraction_from_behavioral(tdf, pid, data_dir)
        if 'At_distraction' not in tdf.columns:
            continue
        tdf = tdf.copy()
        tdf['Participant'] = pid
        frames.append(tdf)
    if not frames:
        print(
            '   Skipping cohort plot4: no usable trial_rt '
            '(need behavioral CSVs under Level*/Results_CSV for row alignment).'
        )
        return

    all_df = pd.concat(frames, ignore_index=True)
    sub = all_df[(all_df['Set'].isin([2, 3])) & (all_df['At_distraction'] == 1)].dropna(
        subset=['RT']
    )
    if sub.empty:
        print('   Skipping cohort plot4: no distraction-trial rows.')
        return

    pp = (
        sub.groupby(['Participant', 'Level', 'Set'], as_index=False)['RT']
        .mean()
    )
    cohort = pp.groupby(['Level', 'Set'], as_index=False)['RT'].agg(
        mean='mean',
        sem=lambda s: s.std(ddof=1) / np.sqrt(len(s)) if len(s) > 1 else 0.0,
    )

    levels = [1, 2, 3]
    x = np.arange(len(levels), dtype=float)
    width = 0.32
    gap = 0.02
    fig, ax = plt.subplots(figsize=(5.2, 4.2))
    offsets = [-width / 2 - gap / 2, width / 2 + gap / 2]
    set_ids = [2, 3]
    labels = ['Set 2 (active)', 'Set 3 (passive)']
    for set_id, dx, leg in zip(set_ids, offsets, labels):
        means, sems = [], []
        for lv in levels:
            row = cohort[(cohort['Level'] == lv) & (cohort['Set'] == set_id)]
            if row.empty:
                means.append(np.nan)
                sems.append(0.0)
            else:
                means.append(float(row['mean'].iloc[0]))
                sems.append(float(row['sem'].iloc[0]))
        cond = 'Distraction+Response' if set_id == 2 else 'Distraction+NoResponse'
        color = COND_COLORS.get(cond, '#333333')
        pos = x + dx
        ax.bar(
            pos,
            means,
            width,
            yerr=sems,
            capsize=2,
            error_kw={'linewidth': 0.9, 'ecolor': '#333'},
            label=leg,
            color=color,
            edgecolor='white',
            linewidth=0.6,
        )

    ax.set_xticks(x)
    ax.set_xticklabels([f'L{lv}' for lv in levels])
    ax.set_xlabel('Level')
    ax.set_ylabel('Mean RT at distraction (s)')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.legend(fontsize=8, framealpha=0.9)
    plt.tight_layout()
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, 'plot4_cohort_rt_at_distraction.png')
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'   Saved: {out_path}')


# ── Plot 6: Recovery shapes ───────────────────────────────────────────────────
def plot6_recovery_shapes(shape_df, out_dir):
    if shape_df is None or shape_df.empty:
        print("   Skipping plot6 (no shape data)")
        return
    cond_map = {2: 'Active Distraction', 3: 'Passive Distraction'}
    shape_data = {}
    for _, row in shape_df.iterrows():
        cond = cond_map.get(row['Set'], f"Set {row['Set']}")
        shape_data.setdefault(cond, {})
        shape_data[cond][row['Shape']] = shape_data[cond].get(row['Shape'], 0) + row['Count']
    if not shape_data:
        return
    fig, axes = plt.subplots(1, len(shape_data), figsize=(5 * len(shape_data), 5))
    if len(shape_data) == 1:
        axes = [axes]
    for ax, (cond, counts) in zip(axes, shape_data.items()):
        filtered = {k: v for k, v in counts.items() if v > 0}
        if not filtered:
            continue
        ax.pie(list(filtered.values()),
               labels=[SHAPE_LABELS.get(k, k) for k in filtered],
               colors=[SHAPE_COLORS.get(k, '#BDC3C7') for k in filtered],
               autopct='%1.0f%%', startangle=90, textprops={'fontsize': 8})
        ax.set_title(f'{cond}\nRecovery Shapes', fontweight='bold')
    fig.suptitle('Recovery Shape Distribution', fontsize=12, fontweight='bold')
    plt.tight_layout()
    save(fig, os.path.join(out_dir, 'plot6_recovery_shapes.png'))


# ── Plot 7: Cumulative debt ───────────────────────────────────────────────────
def plot7_debt(debt_df, out_dir):
    if debt_df is None or debt_df.empty:
        print("   Skipping plot7 (no debt data)")
        return
    cond_map = {1: ('Baseline', '#4C9BE8'),
                2: ('Active Distraction', '#E84C4C'),
                3: ('Passive Distraction', '#E8A94C')}
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    fig.suptitle('Cumulative Attentional Debt', fontsize=13, fontweight='bold')
    for ax_idx, level_id in enumerate([1, 2, 3]):
        ax = axes[ax_idx]
        for set_id in [1, 2, 3]:
            grp = debt_df[(debt_df['Level'] == level_id) & (debt_df['Set'] == set_id)]
            if grp.empty:
                continue
            label, color = cond_map.get(set_id, (str(set_id), '#95A5A6'))
            by_trial = grp.groupby('Trial')['Debt_s'].mean()
            x = by_trial.index.values
            y = by_trial.values
            ax.plot(x, y, color=color, label=label, linewidth=2)
            std = grp.groupby('Trial')['Debt_s'].std().fillna(0).values
            ax.fill_between(x, y - std, y + std, alpha=0.1, color=color)
        ax.set_title(f'Level {level_id}', fontweight='bold')
        ax.set_xlabel('Trial')
        if ax_idx == 0:
            ax.set_ylabel('Cumulative Debt (s)')
        ax.grid(alpha=0.3, linestyle='--')
        ax.legend(fontsize=8)
    plt.tight_layout()
    save(fig, os.path.join(out_dir, 'plot7_cumulative_debt.png'))


# ── Plot 8: Expanded metrics radar ────────────────────────────────────────────
def plot8_radar(df, out_dir):
    metrics = ['IES', 'Post_Error_Slowing', 'Error_Burstiness',
               'Micro_Lapse_Rate', 'RT_Slope']
    available = [m for m in metrics if m in df.columns]
    if not available:
        return
    labels = ['Inv.Efficiency\n(IES)', 'Post-Error\nSlowing',
              'Error\nBurstiness', 'Micro-Lapse\nRate', 'RT\nSlope'][:len(available)]
    conditions = list(COND_COLORS)
    norm_df = df.copy()
    for m in available:
        col = norm_df[m].abs()
        mn, mx = col.min(), col.max()
        norm_df[m] = (col - mn) / (mx - mn) if mx > mn else 0.5

    N = len(available)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + [0]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw=dict(polar=True))
    fig.suptitle('Expanded Behavioral Metrics (Normalized)', fontsize=12, fontweight='bold')
    for ax_idx, level_id in enumerate([1, 2, 3]):
        ax = axes[ax_idx]
        level_norm = norm_df[norm_df['Level'] == level_id]
        for cond in conditions:
            cond_data = level_norm[level_norm['Condition'] == cond]
            if cond_data.empty:
                continue
            vals = [cond_data[m].mean() if m in cond_data.columns else 0
                    for m in available] + [0]
            vals[-1] = vals[0]
            color = COND_COLORS[cond]
            ax.plot(angles, vals, color=color, linewidth=2, label=cond)
            ax.fill(angles, vals, color=color, alpha=0.15)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_title(f'Level {level_id}', fontweight='bold', pad=15)
        ax.set_ylim(0, 1)
        ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=7)
    plt.tight_layout()
    save(fig, os.path.join(out_dir, 'plot8_expanded_metrics_radar.png'))


# ── Plot 10: Distraction cost trajectory ──────────────────────────────────────
def plot10_cost(cost_df, out_dir):
    if cost_df is None or cost_df.empty:
        print("   Skipping plot10 (no cost data)")
        return
    cond_colors = {2: '#E84C4C', 3: '#E8A94C'}
    cond_labels = {2: 'Active Distraction', 3: 'Passive Distraction'}
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    fig.suptitle('Distraction Cost Trajectory (↑=fatigue, ↓=habituation)',
                 fontsize=12, fontweight='bold')
    for ax_idx, level_id in enumerate([1, 2, 3]):
        ax = axes[ax_idx]
        for set_id in [2, 3]:
            grp = cost_df[(cost_df['Level'] == level_id) & (cost_df['Set'] == set_id)]
            if grp.empty:
                continue
            color = cond_colors[set_id]
            by_event = grp.groupby('Event')['Cost_s'].mean()
            x = by_event.index.values
            y = by_event.values
            ax.plot(x, y, color=color, label=cond_labels[set_id], linewidth=2, marker='o', markersize=4)
            std = grp.groupby('Event')['Cost_s'].std().fillna(0).values
            ax.fill_between(x, y - std, y + std, alpha=0.12, color=color)
        ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
        ax.set_title(f'Level {level_id}', fontweight='bold')
        ax.set_xlabel('Distraction Event #')
        if ax_idx == 0:
            ax.set_ylabel('RT Cost (s)')
        ax.grid(alpha=0.3, linestyle='--')
        ax.legend(fontsize=8)
    plt.tight_layout()
    save(fig, os.path.join(out_dir, 'plot10_distraction_cost_trajectory.png'))


# ── BAPS comparison summary ───────────────────────────────────────────────────
def plot_baps_summary(df, out_dir):
    """Single figure comparing BAPS_s, BAPS_3CC, BAPS_Extended side by side."""
    scores = [c for c in ['BAPS_s', 'BAPS_3CC', 'BAPS_Extended'] if c in df.columns]
    if not scores:
        return
    dist = df[df['Condition'] != 'Baseline'].copy()
    if dist.empty:
        return

    fig, axes = plt.subplots(1, len(scores), figsize=(6 * len(scores), 5), sharey=True)
    if len(scores) == 1:
        axes = [axes]
    fig.suptitle('BAPS Score Comparison (Distraction Conditions Only)',
                 fontsize=13, fontweight='bold')
    for ax, score in zip(axes, scores):
        by_level = dist.groupby('Level')[score].mean()
        colors = [SET_COLORS.get(l, '#95A5A6') for l in by_level.index]
        ax.bar([f'L{l}' for l in by_level.index], by_level.values,
               color=colors, alpha=0.85, edgecolor='white', linewidth=0.8)
        ax.axhline(100, color='black', linestyle='--', linewidth=0.8, alpha=0.5,
                   label='Baseline ref')
        ax.set_title(score, fontweight='bold')
        ax.set_xlabel('Level')
        if ax == axes[0]:
            ax.set_ylabel('Mean Score (%)')
        ax.set_ylim(0, 130)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        for i, (level, val) in enumerate(by_level.items()):
            ax.text(i, val + 1, f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    plt.tight_layout()
    save(fig, os.path.join(out_dir, 'plot_baps_comparison.png'))


# ── Main ──────────────────────────────────────────────────────────────────────


In [ ]:
def main():
    parser = argparse.ArgumentParser(description='Generate BAPS plots from CSV files')
    parser.add_argument(
        'results_csv',
        nargs='?',
        default=None,
        help='Path to BAPS_Subject{N}_results.csv (omit with --cohort-plot4)',
    )
    parser.add_argument('--out', help='Output folder (default: plots_Subject{N}/ next to CSV)')
    parser.add_argument(
        '--cohort-plot4',
        action='store_true',
        help='Only write cohort RT-at-distraction bar plot from all BAPS_*_trial_rt.csv',
    )
    parser.add_argument(
        '--data-dir',
        default=None,
        help='Directory to scan for trial CSVs (default: directory of results_csv, or cwd)',
    )
    args = parser.parse_args()

    if args.cohort_plot4:
        data_dir = os.path.abspath(args.data_dir or os.getcwd())
        out_dir = os.path.join(data_dir, 'Analysis_Results', 'cohort_trajectory')
        print(f'Cohort plot4 — scanning: {data_dir}')
        plot4_cohort_rt_at_distraction(data_dir, out_dir)
        return

    csv_path = args.results_csv
    if not csv_path:
        parser.error('results_csv is required unless --cohort-plot4 is set')
    if not os.path.exists(csv_path):
        print(f"Error: file not found: {csv_path}")
        sys.exit(1)

    base_dir = os.path.dirname(os.path.abspath(csv_path))
    base_name = os.path.basename(csv_path)           # BAPS_Subject2_results.csv
    subject_tag = base_name.replace('_results.csv', '')  # BAPS_Subject2

    out_dir = args.out or os.path.join(base_dir, f"plots_{subject_tag.replace('BAPS_', '')}")
    os.makedirs(out_dir, exist_ok=True)
    print(f"Reading:  {csv_path}")
    print(f"Saving to: {out_dir}/")

    # Load main results
    df = pd.read_csv(csv_path)

    # Load optional extra data
    def _load(suffix):
        p = os.path.join(base_dir, f"{subject_tag}{suffix}")
        if os.path.exists(p):
            print(f"  Found: {p}")
            return pd.read_csv(p)
        return None

    trial_df  = _load('_trial_rt.csv')
    if trial_df is not None:
        sm = re.search(r'Subject(\d+)', subject_tag)
        if sm:
            trial_df = _backfill_at_distraction_from_behavioral(
                trial_df, int(sm.group(1)), base_dir
            )
    debt_df   = _load('_debt.csv')
    cost_df   = _load('_cost.csv')
    shape_df  = _load('_shapes.csv')
    print(f"\nGenerating plots...")
    saved = 0
    for name, fn in [
        ('plot1',          lambda: plot1_baps_by_condition(df, out_dir)),
        ('plot1b',         lambda: plot1b_baps3cc_by_condition(df, out_dir)),
        ('plot2',          lambda: plot2_heatmap(df, out_dir)),
        ('plot3',          lambda: plot3_recovery_time(df, out_dir)),
        ('plot3b',         lambda: plot3b_trials_3cc(df, out_dir)),
        ('plot4',          lambda: plot4_trajectory(trial_df, out_dir)),
        ('plot6',          lambda: plot6_recovery_shapes(shape_df, out_dir)),
        ('plot7',          lambda: plot7_debt(debt_df, out_dir)),
        ('plot8',          lambda: plot8_radar(df, out_dir)),
        ('plot10',         lambda: plot10_cost(cost_df, out_dir)),
        ('baps_summary',   lambda: plot_baps_summary(df, out_dir)),
    ]:
        try:
            fn()
            saved += 1
        except Exception as e:
            import traceback
            print(f"   Warning: {name} failed — {e}")
            traceback.print_exc()

    pngs = [f for f in os.listdir(out_dir) if f.endswith('.png')]
    print(f"\nDone: {len(pngs)} PNG files in {out_dir}/")
    for p in sorted(pngs):
        print(f"  {p}")


In [ ]:
# Run this cell to execute the script
main()
